## Organizing the metadata for the WMB dataset from Allen Brain Atlas.
Identifying the vascular clusters and their corresponding anatomical regions. This will help in understanding the distribution of vascular cell types across different brain regions and in performing downstream analyses such as differential expression analysis.


In [ ]:
## imports module
import anndata as ad
import scanpy as sc
import gc, os
import sys
from rich import print
import numpy as np
import psutil
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sea
import harmonypy as hm
import psutil, gc

from rich import print

In [ ]:
## general setting
sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80)
pd.set_option('display.max_columns', None)

# check how many memory used
print(psutil.virtual_memory())
# Force garbage collection
gc.collect()  
sc.settings.set_figure_params(dpi=200, facecolor="white")
PATH = "/home/kibr/Work/Vascular_atlas"

## set path and parameters
os.chdir(PATH)
sc.set_figure_params(figsize=(6, 6), frameon=False)

In [ ]:
## Read in the separated metadata for the Allen WMB dataset
cell_meta = pd.read_csv(PATH + "/Data/Allen_WMB/cell_metadata.csv")
display(cell_meta.tail())

## Check some f the metadata columns
print(cell_meta['region_of_interest_acronym'].nunique()) ## contains 29 regions
print(cell_meta['cluster_alias'].nunique()) ## contain 5322 clusters
print(cell_meta['dataset_label'].value_counts()) ## contain V3/V2/Multi dataset. focusing on V3 and V2 for now. 

In [ ]:
## Read in the clustering annotation excel file for the Allen WMB dataset
cluster_anno = pd.read_excel(PATH + "/Data/Allen_WMB/cl.df_CCN202307220.xlsx", sheet_name="cluster_annotation")
display(cluster_anno.head())

## check the class_label column
# print(cluster_anno['class_label'].value_counts()) ## contain 19 vascular clusters

## subset the cluster annotation to only include vascular clusters
vascular_cluster_anno = cluster_anno[cluster_anno['class_label'].str.contains("Vascular")].copy()
### Then, check the vascular_cluster_anno cl and subclass_label, subttype_label columns
print(vascular_cluster_anno['class_label'].value_counts()) ## contain 19 vascular clusters
print(vascular_cluster_anno['subclass_label'].value_counts()) ## contain 5 subclass vascular
print(vascular_cluster_anno['supertype_label'].value_counts()) ## contain 8 vascular supertypes

In [ ]:
## given the vascular_cluster_anno, subset the cell_meta to only include the vascular clusters
vascular_cell_meta = cell_meta[cell_meta['cluster_alias'].isin(vascular_cluster_anno['cl'].to_list())].copy()
print(vascular_cell_meta.shape)
print(vascular_cell_meta['dataset_label'].value_counts()) ## contain V3/V2/Multi dataset. focusing on V3 and V2 for now.
print(vascular_cell_meta['cluster_alias'].nunique()) ##contain 19 clusters
print(vascular_cell_meta['cluster_alias'].value_counts()) ## contain 29 regions
print(vascular_cell_meta['donor_genotype'].value_counts())
print(vascular_cell_meta['region_of_interest_acronym'].value_counts())
print(vascular_cell_meta['region_of_interest_acronym'].nunique())

In [ ]:
## Add the cell class/subclass/supertype annotation to the vascular_cell_meta
vascular_cell_meta = vascular_cell_meta.merge(vascular_cluster_anno[['cl', 'class_label', 'subclass_label', 'supertype_label']], left_on='cluster_alias', right_on='cl', how='left')
vascular_cell_meta.drop(columns=['cl'], inplace=True)
display(vascular_cell_meta.head())  

## final check how many nuclei in the vascular_cell_meta
print(vascular_cell_meta.shape)

## save the vascular_cell_meta as csv file
vascular_cell_meta.to_csv(PATH + "/Data/Allen_WMB/vascular_cell_metadata.csv", index=False) 

## Read in the raw adata object for the Allen WMB dataset and subset the adata to only include the vascular clusters

In [ ]:
DATA_DIR = os.path.join(PATH, "Data", "Allen_WMB")

# all h5ad files
h5ad_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".h5ad")])
print(h5ad_files)

# build a fast lookup set once
vascular_cells = set(vascular_cell_meta["cell_label"].astype(str).tolist())

adatas = []

## create an loop to read in each file in h5ad_files; then subset each h5ad to only include the vascular clusters; then merge all vascular h5ads together; then save the merged adata as h5ad file.
for fn in h5ad_files:
    fp = os.path.join(DATA_DIR, fn)
    print(f"Processing file: {fn}")

    a = sc.read_h5ad(fp)
    print(f"Original shape of {fn}: {a.shape}")

    # make sure cell ids match the type used in vascular_cell_meta
    a.obs["cell_label"] = a.obs_names.astype(str)

    # subset
    keep = a.obs["cell_label"].isin(vascular_cells).values
    a = a[keep].copy()
    print(f"Subsetted shape of {fn}: {a.shape}")

    # add a batch/source column (useful after merging)
    a.obs["source_file"] = fn

    adatas.append(a)

    del a
    gc.collect()

# merge (row-bind cells)
adata_merged = ad.concat(
    adatas,
    axis=0,
    join="outer",
    label="batch",
    keys=[os.path.splitext(f)[0] for f in h5ad_files],  # nicer batch names
    index_unique="-",  # avoid duplicate obs_names collisions
)

print("Merged shape:", adata_merged.shape)

In [ ]:
## add in the annotation information from the vascular_cell_meta to the adata_merged.obs
meta_cols = ['cell_label', 'class_label', 'subclass_label', 'supertype_label',
             'region_of_interest_acronym', 'dataset_label','donor_label','donor_sex',
             'library_method', 'barcoded_cell_sample_label', 'feature_matrix_label','x','y']

obs = adata_merged.obs.copy()

# remove ambiguity: index -> column with a different name
obs = obs.reset_index(names="obs_name")
obs["cell_label"] = obs["cell_label"].astype(str)

meta = vascular_cell_meta[meta_cols].copy()
meta["cell_label"] = meta["cell_label"].astype(str)

obs = obs.merge(meta, on="cell_label", how="left")

# restore original obs index
obs = obs.set_index("obs_name")
adata_merged.obs = obs

In [ ]:
## Save the merged adata as h5ad file
adata_merged.write_h5ad(PATH + "/Data/Allen_WMB/Allen_WMB_merged_vascular.h5ad") 

## ----- Reloading the merged adata ------ ##

In [ ]:
### ---- Reload the merged adata and prepceossing for downstream analysis
adata = sc.read_h5ad(PATH + "/Data/Allen_WMB/Allen_WMB_merged_vascular.h5ad")

print(adata.shape)
print(adata.obs['region_of_interest_acronym'].nunique())
adata.obs.head()

# check how many memory used
print(psutil.virtual_memory())
# Force garbage collection
gc.collect()  

## Update the gene annotataion to adata.var

In [ ]:
## add varible annotation (gene annotation)
gene_anno = pd.read_csv(PATH + "/Data/Allen_WMB/gene.csv")

# 2) keep original var_names as gene_ids (string)
adata.var["gene_ids"] = adata.var_names.astype(str)

# 3) build id -> symbol mapping (handle potential duplicates in gene_anno)
#    If gene_identifier is unique already, this is trivial.
id2sym = (
    gene_anno[["gene_identifier", "gene_symbol"]]
    .dropna(subset=["gene_identifier"])
    .drop_duplicates(subset=["gene_identifier"], keep="first")
    .set_index("gene_identifier")["gene_symbol"]
)

# 4) annotate symbols in-place (order preserved)
adata.var["gene_symbol"] = adata.var["gene_ids"].map(id2sym)

# 5) choose what to do when symbol is missing:
#    fallback to gene_id so every feature has a name
adata.var["gene_symbol"] = adata.var["gene_symbol"].fillna(adata.var["gene_ids"])

# 6) set var_names to gene_symbol and make unique
adata.var_names = adata.var["gene_symbol"].astype(str)
adata.var_names_make_unique()  # resolves duplicates by appending -1, -2, ...

# 7) (optional) keep clean string dtype
adata.var_names = adata.var_names.astype("string")

print(adata.var.head())

In [ ]:
## Check the cell type information and region information in the adata.obs
print(adata.obs['subclass_label'].value_counts())
print(adata.obs['region_of_interest_acronym'].value_counts())

## Filter the adata to only include the donor_label with more than 10 cells
print(adata.obs['donor_label'].nunique())
print(adata.obs['donor_label'].value_counts())
adata = adata[adata.obs['donor_label'].isin(adata.obs['donor_label'].value_counts()[adata.obs['donor_label'].value_counts() > 10].index)].copy()
print(adata.shape)  

print(adata.obs['donor_label'].nunique())
print(adata.obs['donor_label'].value_counts())

In [ ]:
## save backup
adata.raw = adata.copy()

## filter the features again
sc.pp.filter_genes(adata, min_cells=20) 

print(adata)
print(adata.X[:10,:10])

In [ ]:
## normalization
sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X.copy()

print(adata.X[:8,:8])

## to reach the best results, using the below setting
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    subset=False,
    layer="log1p",
    flavor="seurat",
    batch_key="donor_label",
)

## only using the highly variable genes for integration task
adata = adata[:,adata.var["highly_variable"]].copy()
print(adata)

In [ ]:
### pca and integration
print("Running PCA")
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50, svd_solver="arpack")

## clean up memory before running harmony
print(psutil.virtual_memory())
gc.collect()

In [ ]:
## Harmony integration 
print(adata)
print("Applying Harmony integration")
sc.external.pp.harmony_integrate(adata, key='donor_label', max_iter_harmony=50,basis='X_pca')
rep = "X_pca_harmony"

print(psutil.virtual_memory())

In [ ]:
print("Computing neighbors")
rep = "X_pca_harmony"
sc.pp.neighbors(adata, use_rep=rep,metric="cosine")

print("Computing leiden")
sc.tl.leiden(adata,resolution=0.5,key_added='leiden_harmony')

# Check integration figures for remaining batch(individual effect)
sc.settings.figdir = f"{PATH}/Results/Revision_2/Figures"
os.makedirs(sc.settings.figdir, exist_ok=True)

sc.tl.umap(
    adata,
    # reuse existing neighbors graph
    random_state=42,
    key_added="umap_harmony"
)

sc.pl.embedding(
    adata,
    basis="umap_harmony", 
    color=["leiden_harmony",'subclass_label','region_of_interest_acronym','donor_sex'],
    frameon=False,
    ncols=4,
    size=20,
    legend_loc="on data",
    save=f"_Allen_Vascular_harmony.svg"
)

In [ ]:
## clean up memory before running harmony
gc.collect()
print(psutil.virtual_memory())

In [ ]:
### ---------------------------------------------------------------------
### Saving processed data
### ---------------------------------------------------------------------
adata = adata.raw.to_adata()

print(adata)
print(adata.X[:10,:10])

## set the gene_symbol as the index of adata.var
adata.var.set_index('gene_symbol', inplace=True)
adata.var['gene_names'] = adata.var.index.copy()

## Saving h5ad
results_file = PATH+"/Results/Revision_2/Allen_vascular_processed.h5ad"
adata.write(results_file)
print("Done. Saved:", results_file)

In [ ]:
# ### Given some known cell type markers, annotate the clusters
adata = adata.raw.to_adata()
adata.raw = adata.copy()

print(adata.var.shape)
print(adata.X[:10,:10])

## Normalization and scaling before plotting the marker expression
sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)
sc.pp.scale(adata, max_value=10)

## set var_names to gene_names and make unique
adata.var_names = adata.var["gene_names"].astype(str)
adata.var_names_make_unique()

# markers = {'Large_Artery':'Dkk2','Arterial':"Vegfc",'Capillary':"Slc7a5",'Venous':"Ramp3",'Fenestrated':'Plvap','EndoMT':'Mki67'}
markers = {'SMC':"Tagln",'Pericyte':"Rgs5",'OPC':'Lhfpl3','Astrocyte':"Aqp4",'Neurons':"Syt1",'Arterial':"Vegfc",'Capillary':"Slc7a5",'Venuous':"Ramp3",'Fibroblast':"Cemip",'Microglia':"P2ry12",'Fenestrated':'Plvap','EndoMT':'Mki67','Oligo':'St18'}
sc.pl.dotplot(adata, markers, groupby='leiden_harmony',use_raw=False)

## Repeat the above steps for only endothelial cells in the adata

In [ ]:
## subsetting the adata to only include the endothelial cells
results_file = PATH+"/Results/Revision_2/Allen_Endothelial_processed_clean.h5ad"
adata = sc.read_h5ad(results_file)
adata = adata[adata.obs['subclass_label'] == 'Endo NN'].copy()

## Checking the shape and format
print(adata.shape)
print(adata.X[:10,:10])

# check how many memory used
print(psutil.virtual_memory())
# Force garbage collection
gc.collect()
print(psutil.virtual_memory())

In [ ]:
## Filter the adata to only include the donor_label with more than 10 cells
print(adata.obs['donor_label'].nunique())
# print(adata.obs['donor_label'].value_counts())
adata = adata[adata.obs['donor_label'].isin(adata.obs['donor_label'].value_counts()[adata.obs['donor_label'].value_counts() > 10].index)].copy()
print(adata.shape)  

print(adata.obs['donor_label'].nunique())
print(adata.obs['donor_label'].value_counts())

In [ ]:
## use only the homologuous genes between human and mouse for the downstream analysis
homologous_genes = pd.read_csv(PATH + "/Data/human_mouse_genes.csv")
homologous_genes = homologous_genes[homologous_genes['Mouse gene name'].isin(adata.var_names)].copy()
print(homologous_genes.shape)
print(homologous_genes.head())

In [ ]:
## only keep the homologous genes in the adata
adata = adata[:, adata.var_names.isin(homologous_genes['Mouse gene name'])].copy()
print(adata.shape)
print(adata)

In [ ]:
## save backup
adata.raw = adata.copy()

## filter the features again
sc.pp.filter_genes(adata, min_cells=20) ## for endothelial cells, around 21600 genes were kept

print(adata)
print(adata.X[:10,:10])

In [ ]:
## normalization
sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X.copy()

print(adata.X[:8,:8])

In [ ]:
print(adata.obs['donor_label'].nunique())
# print(adata.obs['donor_label'].value_counts())

In [ ]:
## set var_names to gene_names and make unique
adata.var_names = adata.var["gene_names"].astype(str)
adata.var_names_make_unique()

## to reach the best results, using the below setting
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=5000,
    subset=False,
    layer="log1p",
    flavor="seurat",
    batch_key="donor_label",
)

## only using the highly variable genes for integration task
adata = adata[:,adata.var["highly_variable"]].copy()
print(adata)

In [ ]:
### pca and integration
print("Running PCA")
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50, svd_solver="arpack")
print(psutil.virtual_memory())

In [ ]:
## Harmony integration 
print(adata)
print("Applying Harmony integration")
sc.external.pp.harmony_integrate(adata, key='donor_label', max_iter_harmony=50,basis='X_pca')
rep = "X_pca_harmony"

print(psutil.virtual_memory())

In [ ]:
print("Computing neighbors")
rep = "X_pca_harmony"
sc.pp.neighbors(adata, use_rep=rep,metric="cosine")

print("Computing leiden")
sc.tl.leiden(adata,resolution=0.5,key_added='leiden_harmony')

# Check integration figures for remaining batch(individual effect)
sc.settings.figdir = f"{PATH}/Results/Revision_2/Figures"
os.makedirs(sc.settings.figdir, exist_ok=True)

sc.tl.umap(
    adata,
    # reuse existing neighbors graph
    random_state=42,
    key_added="umap_harmony"
)

sc.pl.embedding(
    adata,
    basis="umap_harmony", 
    color=["leiden_harmony",'subclass_label','region_of_interest_acronym','donor_sex','Cell_type'],
    frameon=False,
    ncols=4,
    size=20,
    legend_loc="on data",
    # save=f"_Allen_Endothelial_harmony.svg"
)

In [ ]:
gc.collect()
print(psutil.virtual_memory())

In [ ]:
### ---------------------------------------------------------------------
### Saving processed data
### ---------------------------------------------------------------------
adata = adata.raw.to_adata()

print(adata)
print(adata.X[:10,:10])

## set the gene_symbol as the index of adata.var
adata.var.set_index('gene_names', inplace=True)
adata.var['gene_names'] = adata.var.index.copy()

## Saving h5ad
# results_file = PATH+"/Results/Revision_2/Allen_Endothelial_processed.h5ad" ## First round
results_file = PATH+"/Results/Revision_2/Allen_Endothelial_processed_clean_homologous.h5ad" ## Removal of clusters of potential doublets and contaminated clusters and only keep the homologous genes between human and mouse for the downstream analysis
adata.write(results_file)
print("Done. Saved:", results_file)

### Check marker genes of the endothelial clusters

In [ ]:
# adata = sc.read_h5ad(PATH + "/Results/Revision_2/Allen_Endothelial_processed.h5ad")
adata = sc.read_h5ad(PATH + "/Results/Revision_2/Allen_Endothelial_processed_clean_homologous.h5ad")
print(adata)
print(adata.X[:10,:10])

In [ ]:
## check how many brain reigon included in the adata
print(adata.obs['region_of_interest_acronym'].nunique())

In [ ]:
print("Computing leiden")
sc.tl.leiden(adata,resolution=0.8,key_added='leiden_harmony')

In [ ]:
### Given some known cell type markers, annotate the clusters
adata.raw = adata.copy()

print(adata)
print(adata.X[:10,:10])

## normalization
sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)
sc.pp.scale(adata, max_value=10)

## set var_names to gene_names and make unique
adata.var_names = adata.var["gene_names"].astype(str)
adata.var_names_make_unique()

##check marker genes of endothelial subtypes
markers = {
    'Arterial': ['Vegfc', 'Dkk2','Arl15'],
    'Capillary': ['Slc7a5', 'Slc2a1', 'Cldn5'],
    'Venuous': ['Ramp3', 'Tshz2'],
    'Fenestrated': ['Plvap'],
    'EndoMT': ['Mki67', 'Angpt2'],
    # 'SMC': ['Tagln', 'Acta2', 'Myh11'],
    # 'Pericyte': ['Rgs5', 'Pdgfrb'],
    # 'OPC': ['Lhfpl3', 'Pdgfra'],
    # 'Astrocyte': ['Aqp4', 'Gfap', 'Slc1a2'],
    # 'Neurons': ['Syt1', 'Snap25', 'Rbfox3'],
    # 'Fibroblast': ['Cemip', 'Col1a1', 'Col1a2', 'Lama2'],
    # 'Microglia': ['P2ry12', 'Cx3cr1', 'Tmem119'],
    # 'Oligo': ['St18', 'Mbp']
}
# markers = {'SMC':"Tagln",'Pericyte':"Rgs5",'OPC':'Lhfpl3','Astrocyte':"Aqp4",'Neurons':"Syt1",'Arterial':"Vegfc",'Capillary':"Slc7a5",'Venuous':"Ramp3",'Fibroblast':"Cemip",'Microglia':"P2ry12",'Fenestrated':'Plvap','EndoMT':'Mki67','Oligo':'St18'}
sc.pl.dotplot(adata, markers, groupby='leiden_harmony',use_raw=False)

In [ ]:
## Assign cell type annotation based on the dotplot and known markers
sc.settings.figdir = f"{PATH}/Results/Revision_2/Figures"
# cell types
cluster2celltype = {
    "0": "Capillary",
    "1": "Venous",
    "2": "Arterial",
    "3": "Capillary",
    "4": "Arterial",
    "5": "Venous",
    "6": "Capillary",
    "7": "Capillary",
    "8": "Capillary",
    "9": "Capillary",
    "10": "Capillary",
    "11": "Capillary",
    "12": "Fenestrated_Capillary"
}

adata.obs["Cell_type"] = adata.obs["leiden_harmony"].map(cluster2celltype)


sc.pl.embedding(
    adata,
    basis='umap_harmony',   # <- THIS picks .obsm['X_' + key]
    color=['leiden_harmony','Cell_type'],
    frameon=False,
    ncols=4,
    size=20,
    legend_loc="on data",
    use_raw = False,
    # save=f"_Endo_CT.svg"
    )

In [ ]:
## plottint the celtype with specific color map
sc.settings.set_figure_params(figsize=(8, 6), frameon=False)

celltype_colors = {
    # 'Large_Artery': '#B2182B',
    'Arterial': '#F4A582',
    'Capillary': '#66C2A5',
    'Venous': '#4393C3',
    'Fenestrated_Capillary': '#FDAE61',
    # 'EndoMT': '#BC80BD'
}

# enforce category order
adata.obs['Cell_type'] = adata.obs['Cell_type'].cat.reorder_categories(
    celltype_colors.keys(), ordered=True
)

# assign colors
adata.uns['Cell_type_colors'] = list(celltype_colors.values())


sc.pl.embedding(
    adata,
    basis='umap_harmony',   # <- THIS picks .obsm['X_' + key]
    color=['Cell_type'],
    frameon=False,
    ncols=4,
    size=20,
    # legend_loc="on data",
    use_raw = False,
    # save=f"_WMB_Endo_CT_only.svg"
    )

In [ ]:
## Based on the dotplot, we can remove some of the clusters with expression of marker genes of other cell types.
adata = adata.raw.to_adata()
# cluter_removal = ['7', '8', '10']
# adata = adata[~adata.obs['leiden_harmony'].isin(cluter_removal)].copy()
# print(adata)
# print(adata.X[:10,:10])

## set the gene_symbol as the index of adata.var
adata.var.set_index('gene_names', inplace=True)
adata.var['gene_names'] = adata.var.index.copy()

## Saving h5ad
results_file = PATH+"/Results/Revision_2/Allen_Endothelial_processed_clean_homologous.h5ad"
adata.write(results_file)
print("Done. Saved:", results_file)

## Builting region-level representation (Donor-balanced centroids in PC spave)

In [ ]:
## reload the cleaned adata
results_file = PATH+"/Results/Revision_2/Allen_Endothelial_processed_clean_homologous.h5ad"
adata = sc.read_h5ad(results_file)
print(adata)

In [ ]:
# Choose how many PCs to represent each cell
rep = "X_pca_harmony"
n_pcs_use = 30
Xpc = adata.obsm[rep][:, :n_pcs_use]

# centroid for each (region, donor) group
df_index = adata.obs[["region_of_interest_acronym", "donor_label"]].copy()
df_index["cell_idx"] = np.arange(adata.n_obs)

# Compute group centroids efficiently
# We'll build a table of centroids: rows = (region, donor), cols = PC1..PCn
pcs = pd.DataFrame(Xpc, index=adata.obs_names, columns=[f"PC{i+1}" for i in range(n_pcs_use)])
pcs["region_of_interest_acronym"] = adata.obs["region_of_interest_acronym"].values
pcs["donor_label"]  = adata.obs["donor_label"].values

rg_dn_centroids = pcs.groupby(["region_of_interest_acronym", "donor_label"]).mean()
# print(rg_dn_centroids.head())

## donor-balanced region centroid
rg_centroids = rg_dn_centroids.groupby(level=0).mean()

# Optional: track how many donors contribute per region (useful QC)
donors_per_region = rg_dn_centroids.reset_index().groupby("region_of_interest_acronym")["donor_label"].nunique()

## Create region -level AnnData and run leiden on region graph
adata_rg = sc.AnnData(X=rg_centroids.values)
adata_rg.obs_names = rg_centroids.index.astype(str)
adata_rg.var_names = [f"PC{i+1}" for i in range(n_pcs_use)]

adata_rg.obs["n_donors"] = donors_per_region.reindex(adata_rg.obs_names).fillna(0).astype(int).values

print(adata_rg)

In [ ]:
# Build kNN graph across ~40 regions
# With only 40 nodes, keep neighbors modest (e.g., 5–15)
sc.pp.neighbors(adata_rg, n_neighbors=5, n_pcs=n_pcs_use, use_rep="X")

# Leiden clustering of regions (sweep resolution later)
sc.tl.leiden(adata_rg, resolution=1, key_added="leiden")

# 2D embedding for visualization 
sc.tl.umap(adata_rg, min_dist=0.5,random_state=42)

sc.tl.dendrogram(adata_rg, groupby="leiden")

## working on the dot size based on nuclei counts
nuclei_counts = adata.obs['region_of_interest_acronym'].value_counts()

## ordering counts based on adata_rg.obs['region_of_interest_acronym']
nuclei_counts_dict = nuclei_counts.to_dict()

# Size parameters might require scaling; you can adjust the scaling factor as needed
size_factor = 1  # Adjust this factor to change the size scale
dot_sizes = np.array([nuclei_counts_dict[region] * size_factor for region in adata_rg.obs.index])
print(nuclei_counts.head())

In [ ]:
# ----------------------------
# Plotting (region-level)
# ----------------------------
sc.set_figure_params(figsize=(8, 8), frameon=False)

sc.pl.umap(
    adata_rg,
    color=["leiden"],
    size=dot_sizes,
    legend_loc="on data",
    show=False
)

# add region labels
ax = plt.gca()
X = adata_rg.obsm["X_umap"]

for i, region in enumerate(adata_rg.obs_names):
    ax.text(
        X[i, 0],
        X[i, 1],
        region,
        fontsize=8,
        ha="center",
        va="center"
    )

plt.show()

# sc.pl.umap(adata_rg, color=["region_name"],legend_loc = "on data")
sc.pl.dendrogram(adata_rg, groupby="leiden",)

In [ ]:
## adding the anatomical_division_label from adata to adata_rg
# First, create a mapping from region_of_interest_acronym to anatomical_division_label
region_to_anatomical = adata.obs.drop_duplicates(subset=["region_of_interest_acronym", "anatomical_division_label"]).set_index("region_of_interest_acronym")["anatomical_division_label"].to_dict() 
# Then, map the region_of_interest_acronym in adata_rg to anatomical_division_label
adata_rg.obs["anatomical_division_label"] = adata_rg.obs_names.map(region_to_anatomical)
print(adata_rg.obs['anatomical_division_label'].value_counts())
adata_rg.obs['reigon_of_interest_acronym'] = adata_rg.obs_names.copy()
# adata_rg.obs

In [ ]:
## Option 2
# ----------------------------
# 1) Define clusters (leiden -> region_cluster)
# ----------------------------
cluster_map = {
    "0": "Cluster_1",
    "1": "Cluster_2",
    "2": "Cluster_3",
    # "3": "Cluster_3",
}

adata_rg.obs["region_cluster"] = adata_rg.obs["leiden"].astype(str).map(cluster_map).fillna("Unassigned").astype("category")
display(adata_rg.obs)

pd.crosstab(adata_rg.obs["region_cluster"], adata_rg.obs["anatomical_division_label"])

In [ ]:

import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'

# ----------------------------
# 1) Define clusters (leiden -> region_cluster)
# ----------------------------
cluster_map = {
    "0": "Cluster_1",
    "1": "Cluster_2",
    "2": "Cluster_3",
    # "3": "Cluster_3",
}

adata_rg.obs["region_cluster"] = adata_rg.obs["leiden"].astype(str).map(cluster_map).fillna("Unassigned").astype("category")


# ----------------------------
# 2) Create a refined anatomical label that splits STRd from STR
# ----------------------------
adata_rg.obs["anatomical_division_label_refined"] = adata_rg.obs["anatomical_division_label"].astype(str).copy()

# Override STRd rows with its own label
adata_rg.obs.loc[
    adata_rg.obs["reigon_of_interest_acronym"] == "STRd",
    "anatomical_division_label_refined"
] = "STRd"

adata_rg.obs["anatomical_division_label"] = adata_rg.obs["anatomical_division_label_refined"].astype("category")

# ----------------------------
# 3) Define Region_layer colors (your palette)
# ----------------------------
region_color_map = {
    "Isocortex": "#009E73",
    "HPF":       "#03B8D8",
    "MY":        "#D55E00",
    "CB":        "#046299",
    "MB":        "#D55E00",
    "HY":        "#03B8D8",
    "OLF":       "#999999",
    "STR":       "#03B8D8",
    "STRd":      "#915330",   # <-- STRd gets its own color
    "CTXsp":     "#915330",
    "P":         "#D55E00",
    "PAL":       "#03B8D8",    
    "TH":        "#03B8D8",
}



# Ensure categorical for stable color ordering
adata_rg.obs["anatomical_division_label"] = adata_rg.obs["anatomical_division_label"].astype("category")

# Scanpy expects a list of colors aligned to category order
adata_rg.uns["anatomical_division_label_colors"] = [
    region_color_map.get(cat, "#BDBDBD")  
    for cat in adata_rg.obs["anatomical_division_label"].cat.categories
]

# ----------------------------
# 3) Plot UMAP colored by anatomical_division_label
# ----------------------------
sc.set_figure_params(figsize=(9, 9), frameon=False)

sc.pl.umap(
    adata_rg,
    color="anatomical_division_label",
    size=dot_sizes,
    legend_loc="right margin",
    show=False
)

ax = plt.gca()
X = adata_rg.obsm["X_umap"]

# ----------------------------
# 4) Add region labels (after the first "_")
# ----------------------------
for (x, y), region in zip(X, adata_rg.obs_names):
    label = adata_rg.obs[adata_rg.obs_names == region]['reigon_of_interest_acronym'].astype(str).values[0]
    ax.text(
        x, y, label,
        fontsize=10,
        ha="center",
        va="center",
        clip_on=True
    )
# ----------------------------
# 5) Add KDE contours per region_cluster + label each contour
# ----------------------------
clusters = adata_rg.obs["region_cluster"].astype(str)

# Tuning knobs for "closer" contours:
bw_adjust = 1.2   # smaller -> tighter / more detailed
thresh = 0.15     # larger -> tighter (keeps higher-density region)
alpha = 0.1  # smaller -> more transparent contours (can help with overlap)

for cl in sorted(clusters.unique()):
    if cl == "Unassigned":
        continue

    idx = (clusters == cl).values
    pts = X[idx, :]

    # KDE unstable for very small clusters
    if pts.shape[0] < 3:
        continue

    sea.kdeplot(
        x=pts[:, 0],
        y=pts[:, 1],
        fill=True,          # << shaded region
        levels=4,           # single filled region
        bw_adjust=bw_adjust,
        thresh=thresh,
        alpha=alpha,
        linewidth=0,
        ax=ax
    )

    # Optional: cluster label at center
    cx, cy = np.median(pts[:, 0]), np.median(pts[:, 1])
    ax.text(
        cx, cy, cl,
        fontsize=11,
        ha="center",
        va="center",
        bbox=dict(boxstyle="round,pad=0.2", alpha=0.6, linewidth=0),
        clip_on=True
    )

# # Optional cleanup
# ax.set_xticks([])
# ax.set_yticks([])
# ax.set_aspect("equal")

plt.savefig("./Results/Revision_2/Figures/umap_region_clusters_abb_Allen_WMB_Endothelial.svg",format="svg",bbox_inches="tight",dpi=600)
plt.show()

In [ ]:
### Mapping the cluster labels back to the original adata
# First, create a mapping from reigon_of_interest_acronym to region_cluster
region_to_cluster = adata_rg.obs[["reigon_of_interest_acronym", "region_cluster"]].copy().set_index("reigon_of_interest_acronym")["region_cluster"].to_dict()
region_to_leiden = adata_rg.obs[["reigon_of_interest_acronym", "leiden"]].copy().set_index("reigon_of_interest_acronym")["leiden"].to_dict()  
# Then, map the reigon_of_interest_acronym in adata to region_cluster
adata.obs["region_cluster"] = adata.obs["region_of_interest_acronym"].map(region_to_cluster)
adata.obs["region_leiden"] = adata.obs["region_of_interest_acronym"].map(region_to_leiden)
print(adata.obs['region_cluster'].value_counts())   
print(adata.obs['region_leiden'].value_counts())   

In [ ]:
sc.pl.embedding(adata, basis="umap_harmony", color=['leiden_harmony',"region_of_interest_acronym", "region_cluster","region_leiden"],frameon=False, size=20,legend_loc="on data",)

In [ ]:
print(adata)
print(adata.X[:10,:10])

In [ ]:
### ---------------------------------------------------------------------
### Saving processed data
### ---------------------------------------------------------------------
# adata = adata.raw.to_adata()
# print(adata)
## Saving h5ad
results_file = PATH+"/Results/Revision_2/Allen_Endothelial_processed_clean_homologous.h5ad"
adata.write(results_file)
print("Done. Saved:", results_file)

In [ ]:
results_file = PATH+"/Results/Revision_2/Allen_Endothelial_processed_clean_homologous.h5ad"
adata = sc.read_h5ad(results_file)
print(adata)

In [ ]:
cols = ['region_of_interest_acronym','anatomical_division_label','region_cluster']

df_unique = (
    adata.obs[cols]
    .drop_duplicates()
)
display(df_unique)
df_unique.to_csv("./Results/Revision_2/MOUSE_unique_region_metadata_homologous.csv", index=False)

In [ ]:
pd.crosstab(adata.obs["region_cluster"], adata.obs["anatomical_division_label"])